# LeetCode 350 — Intersection of Two Arrays II bằng Python

**Link đề:** https://leetcode.com/problems/intersection-of-two-arrays-ii/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Two Pointers, Binary Search, Sorting

```python
class Solution:
    def intersect(self, nums1: list[int], nums2: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Phân biệt bài này với **349**: lần này **giữ số lần lặp** — phần tử xuất hiện `min(số lần ở nums1, số lần ở nums2)` lần.
2. Học kỹ thuật **đếm tần suất** bằng `dict` và `collections.Counter`.
3. Đi từ cách trực giác (đánh dấu đã dùng) → **hash map đếm** `O(n + m)` → **sắp xếp + hai con trỏ**.
4. Trace trạng thái bộ đếm từng bước; chứng minh tính đúng; Time/Space.
5. Trả lời 3 câu **Follow up** của đề: mảng đã sắp xếp, mảng nhỏ/lớn lệch nhau, dữ liệu nằm trên đĩa.

## 1. Đề bài

Cho hai mảng số nguyên `nums1`, `nums2`. Trả về giao của chúng, trong đó **mỗi phần tử xuất hiện số lần bằng số lần nó có mặt ở cả hai mảng** (tức là `min` của hai số đếm). Thứ tự tùy ý.

| `nums1` | `nums2` | Kết quả | Vì sao |
|---|---|---|---|
| `[1, 2, 2, 1]` | `[2, 2]` | `[2, 2]` | `2` có 2 lần ở `nums1`, 2 lần ở `nums2` → `min(2, 2) = 2` lần |
| `[4, 9, 5]` | `[9, 4, 9, 8, 4]` | `[4, 9]` (hoặc `[9, 4]`) | `4`: min(1, 2)=1; `9`: min(1, 2)=1 |

**Constraints:** `1 <= len(nums1), len(nums2) <= 1000`; `0 <= nums1[i], nums2[i] <= 1000`.

**Follow up:**
1. Nếu mảng đã được sắp xếp thì tối ưu thế nào?
2. Nếu `nums1` nhỏ hơn nhiều so với `nums2` thì thuật toán nào tốt hơn?
3. Nếu `nums2` nằm trên đĩa và bộ nhớ hạn chế, không thể nạp hết một lần?

In [1]:
# So sánh nhanh 349 (duy nhất) và 350 (giữ số lần) trên cùng input
nums1, nums2 = [1, 2, 2, 1], [2, 2]
print("Bài 349 sẽ trả:", sorted(set(nums1) & set(nums2)))
print("Bài 350 phải trả: [2, 2]  vì 2 xuất hiện 2 lần ở mỗi mảng")

Bài 349 sẽ trả: [2]
Bài 350 phải trả: [2, 2]  vì 2 xuất hiện 2 lần ở mỗi mảng


### 1.1. Hiểu "min số lần" bằng tay

`nums1 = [1, 1, 1, 3]`, `nums2 = [1, 3, 3, 1]`:

| Giá trị | Số lần ở `nums1` | Số lần ở `nums2` | Xuất hiện trong kết quả |
|---:|---:|---:|---:|
| 1 | 3 | 2 | `min(3, 2) = 2` |
| 3 | 1 | 2 | `min(1, 2) = 1` |

→ Kết quả (sau khi sắp): `[1, 1, 3]`. Ý nghĩa: mỗi phần tử của một mảng chỉ được **ghép cặp** với **một** phần tử bằng nó của mảng kia.

## 2. Nền tảng Python

### 2.1. `dict` — bảng băm key → value

`dict` lưu cặp **khóa → giá trị**. Ở bài này: khóa là **số**, giá trị là **số lần đã đếm**.

| Thao tác | Cú pháp |
|---|---|
| Tạo rỗng | `cnt = {}` |
| Gán / cập nhật | `cnt[x] = 5` |
| Đọc (lỗi nếu thiếu) | `cnt[x]` → `KeyError` nếu `x` chưa có |
| Đọc an toàn | `cnt.get(x, 0)` → trả `0` nếu thiếu |
| Kiểm tra khóa | `x in cnt` |

Tra cứu/gán trung bình `O(1)`.

In [2]:
cnt = {}
cnt[2] = 1
cnt[2] = cnt[2] + 1
print(cnt, "| cnt[2] =", cnt[2])
print("cnt.get(7, 0) =", cnt.get(7, 0), "| 7 in cnt:", 7 in cnt)
try:
    cnt[7]
except KeyError as e:
    print("Đọc cnt[7] trực tiếp ->", type(e).__name__, e)

{2: 2} | cnt[2] = 2
cnt.get(7, 0) = 0 | 7 in cnt: False
Đọc cnt[7] trực tiếp -> KeyError 7


### 2.2. Mẫu đếm tần suất bằng `dict.get`

```python
cnt[x] = cnt.get(x, 0) + 1
```

Đọc: "lấy số đếm hiện tại của `x` (mặc định 0), cộng 1, ghi lại".

In [3]:
def count_freq(nums):
    cnt = {}
    for x in nums:
        cnt[x] = cnt.get(x, 0) + 1
        print(f"gặp {x} -> cnt = {cnt}")
    return cnt

count_freq([1, 2, 2, 1])

gặp 1 -> cnt = {1: 1}
gặp 2 -> cnt = {1: 1, 2: 1}
gặp 2 -> cnt = {1: 1, 2: 2}
gặp 1 -> cnt = {1: 2, 2: 2}


{1: 2, 2: 2}

### 2.3. `collections.Counter` — bộ đếm có sẵn

`Counter(nums)` là một `dict` con, đếm sẵn tần suất. Đọc khóa thiếu trả **0** (không lỗi). Đặc biệt `c1 & c2` cho **min** từng khóa — đúng định nghĩa của bài!

`elements()` trả về các khóa, mỗi khóa lặp lại đúng số lần đếm.

In [4]:
from collections import Counter
c1 = Counter([1, 1, 1, 3])
c2 = Counter([1, 3, 3, 1])
print("c1 =", c1, "| c2 =", c2)
print("c1[99] =", c1[99], "(khóa thiếu -> 0)")
print("c1 & c2 =", c1 & c2, "(min từng khóa)")
print("list((c1 & c2).elements()) =", list((c1 & c2).elements()))

c1 = Counter({1: 3, 3: 1}) | c2 = Counter({1: 2, 3: 2})
c1[99] = 0 (khóa thiếu -> 0)
c1 & c2 = Counter({1: 2, 3: 1}) (min từng khóa)
list((c1 & c2).elements()) = [1, 1, 3]


### 2.4. `min(a, b)` và `[x] * k`

- `min(3, 2)` → `2`.
- `[x] * k` tạo list gồm `k` bản sao của `x`: `[1] * 2` → `[1, 1]`. Với `k = 0` → `[]`.
- `list.extend(other)` nối cả list `other` vào cuối.

In [5]:
out = []
out.extend([1] * min(3, 2))
out.extend([3] * min(1, 2))
out.extend([7] * 0)
print(out)

[1, 1, 3]


## 3. Cách 1 — Trực giác: ghép cặp và **đánh dấu đã dùng**

Người mới thường nghĩ: "Với mỗi `x` trong `nums1`, tìm một phần tử bằng `x` **chưa dùng** trong `nums2`. Tìm được thì thêm `x` vào kết quả và đánh dấu phần tử đó đã dùng."

Đánh dấu bằng list `used` gồm các `True/False` cùng độ dài với `nums2`.

In [6]:
def intersect_brute(nums1: list[int], nums2: list[int]) -> list[int]:
    used = [False] * len(nums2)
    result = []
    for x in nums1:
        for j in range(len(nums2)):
            if not used[j] and nums2[j] == x:
                used[j] = True          # mỗi phần tử nums2 chỉ ghép 1 lần
                result.append(x)
                break                   # x đã có cặp -> sang x tiếp theo
    return result

print(intersect_brute([1, 2, 2, 1], [2, 2]))
print(intersect_brute([4, 9, 5], [9, 4, 9, 8, 4]))
print(intersect_brute([1, 1, 1, 3], [1, 3, 3, 1]))

[2, 2]
[4, 9]
[1, 1, 3]


**`break`** thoát ngay vòng lặp **gần nhất** (vòng `for j`), quay về vòng `for x`. Nếu thiếu `break`, một `x` có thể "ăn" nhiều phần tử của `nums2` → sai số lần.

### 3.1. Nút thắt

Thời gian `O(n · m)`: mỗi `x` lại quét `nums2` từ đầu. Câu hỏi lặp lại là "**còn bao nhiêu** bản sao của `x` chưa dùng?". Nếu ta **đếm trước** tần suất của một mảng, câu hỏi đó trả lời trong `O(1)`.

## 4. Cách 2 — Hash map đếm (khuyên dùng)

Ba đoạn nhỏ:

**Đoạn 1 — đếm `nums1`:** `cnt[x]` = số bản sao của `x` **còn có thể ghép**.

**Đoạn 2 — quét `nums2`:** với mỗi `y`, nếu `cnt.get(y, 0) > 0` → còn bản sao để ghép → thêm `y` vào kết quả.

**Đoạn 3 — trừ bộ đếm:** `cnt[y] -= 1` vì đã dùng một bản sao. Khi về 0, các `y` sau sẽ không được ghép nữa → tự động đảm bảo `min`.

In [7]:
def intersect_hashmap(nums1: list[int], nums2: list[int]) -> list[int]:
    cnt = {}
    for x in nums1:                 # Đoạn 1
        cnt[x] = cnt.get(x, 0) + 1
    result = []
    for y in nums2:                 # Đoạn 2
        if cnt.get(y, 0) > 0:
            result.append(y)
            cnt[y] -= 1             # Đoạn 3
    return result

print(intersect_hashmap([1, 2, 2, 1], [2, 2]))
print(intersect_hashmap([4, 9, 5], [9, 4, 9, 8, 4]))
print(intersect_hashmap([1, 1, 1, 3], [1, 3, 3, 1]))

[2, 2]
[9, 4]
[1, 3, 1]


**`x -= 1`** là viết tắt của `x = x - 1` (tương tự `+=`).

### 4.1. Mẹo: đếm mảng **nhỏ hơn**

Bộ nhớ của `cnt` tỉ lệ với số giá trị khác nhau của mảng được đếm. Nếu `len(nums1) > len(nums2)`, đổi chỗ hai mảng để đếm mảng nhỏ → `O(min(n, m))` bộ nhớ. Đây cũng là câu trả lời cho **Follow up 2**.

## 5. TRACE từng bước — `nums1 = [1, 1, 1, 3]`, `nums2 = [1, 3, 3, 1]`

Sau Đoạn 1: `cnt = {1: 3, 3: 1}`.

| Bước | `y` | `cnt[y]` trước | Ghép được? | `cnt` sau | `result` |
|---:|---:|---:|:---:|---|---|
| 1 | 1 | 3 | ✔ | `{1: 2, 3: 1}` | `[1]` |
| 2 | 3 | 1 | ✔ | `{1: 2, 3: 0}` | `[1, 3]` |
| 3 | 3 | 0 | ✘ (hết bản sao) | `{1: 2, 3: 0}` | `[1, 3]` |
| 4 | 1 | 2 | ✔ | `{1: 1, 3: 0}` | `[1, 3, 1]` |

Bước 3 là nhánh đáng chú ý: `3` có trong `nums1` nhưng **đã dùng hết** → không thêm. Đây chính là cơ chế tạo ra `min(1, 2) = 1`.

In [8]:
def trace_intersect(nums1, nums2):
    cnt = {}
    for x in nums1:
        cnt[x] = cnt.get(x, 0) + 1
    print("Sau khi đếm nums1: cnt =", cnt)
    result = []
    for step, y in enumerate(nums2, start=1):
        before = cnt.get(y, 0)
        if before > 0:
            result.append(y)
            cnt[y] -= 1
            action = "ghép -> giảm cnt"
        else:
            action = "không ghép (cnt = 0 hoặc không có)"
        print(f"Bước {step}: y={y} | cnt[y] trước={before} | {action} | cnt={cnt} | result={result}")
    return result

_ = trace_intersect([1, 1, 1, 3], [1, 3, 3, 1])

Sau khi đếm nums1: cnt = {1: 3, 3: 1}
Bước 1: y=1 | cnt[y] trước=3 | ghép -> giảm cnt | cnt={1: 2, 3: 1} | result=[1]
Bước 2: y=3 | cnt[y] trước=1 | ghép -> giảm cnt | cnt={1: 2, 3: 0} | result=[1, 3]
Bước 3: y=3 | cnt[y] trước=0 | không ghép (cnt = 0 hoặc không có) | cnt={1: 2, 3: 0} | result=[1, 3]
Bước 4: y=1 | cnt[y] trước=2 | ghép -> giảm cnt | cnt={1: 1, 3: 0} | result=[1, 3, 1]


## 6. Cách 3 — `Counter` và toán tử `&`

Đúng nghĩa đen của đề: giao theo **min số lần**.

```python
return list((Counter(nums1) & Counter(nums2)).elements())
```

In [9]:
def intersect_counter(nums1: list[int], nums2: list[int]) -> list[int]:
    return list((Counter(nums1) & Counter(nums2)).elements())

print(intersect_counter([1, 2, 2, 1], [2, 2]))
print(sorted(intersect_counter([4, 9, 5], [9, 4, 9, 8, 4])))

[2, 2]
[4, 9]


Viết tường minh hơn (không dùng `&`) để thấy rõ `min`:

In [10]:
def intersect_counter_explicit(nums1, nums2):
    c1, c2 = Counter(nums1), Counter(nums2)
    out = []
    for x in c1:                       # duyệt các khóa của c1
        k = min(c1[x], c2[x])          # c2[x] = 0 nếu x không có trong nums2
        out.extend([x] * k)
    return out

print(sorted(intersect_counter_explicit([1, 1, 1, 3], [1, 3, 3, 1])))

[1, 1, 3]


## 7. Cách 4 — Sắp xếp + hai con trỏ (Follow up 1)

Nếu hai mảng **đã sắp xếp**, ta không cần bộ đếm:

- `a[i] < c[j]` → `i += 1` (giá trị `a[i]` quá nhỏ, không còn bạn ghép phía sau `c`).
- `a[i] > c[j]` → `j += 1`.
- Bằng nhau → ghép cặp: thêm vào kết quả, **tăng cả hai** (mỗi phần tử chỉ ghép một lần).

Khác bài 349: **không** chống trùng — mỗi lần bằng nhau là một cặp mới.

In [11]:
def intersect_two_pointers(nums1: list[int], nums2: list[int]) -> list[int]:
    a, c = sorted(nums1), sorted(nums2)
    i = j = 0
    result = []
    while i < len(a) and j < len(c):
        if a[i] < c[j]:
            i += 1
        elif a[i] > c[j]:
            j += 1
        else:
            result.append(a[i])
            i += 1
            j += 1
    return result

print(intersect_two_pointers([1, 2, 2, 1], [2, 2]))
print(intersect_two_pointers([4, 9, 5], [9, 4, 9, 8, 4]))
print(intersect_two_pointers([1, 1, 1, 3], [1, 3, 3, 1]))

[2, 2]
[4, 9]
[1, 1, 3]


In [12]:
def trace_two_pointers(nums1, nums2):
    a, c = sorted(nums1), sorted(nums2)
    print("a =", a, "| c =", c)
    i = j = 0
    result = []
    while i < len(a) and j < len(c):
        if a[i] < c[j]:
            print(f"i={i}, j={j}: {a[i]} < {c[j]} -> i += 1")
            i += 1
        elif a[i] > c[j]:
            print(f"i={i}, j={j}: {a[i]} > {c[j]} -> j += 1")
            j += 1
        else:
            result.append(a[i])
            print(f"i={i}, j={j}: bằng {a[i]} -> ghép, result={result}")
            i += 1
            j += 1
    print(f"Dừng: i={i}/{len(a)}, j={j}/{len(c)} -> result={result}")
    return result

_ = trace_two_pointers([1, 1, 1, 3], [1, 3, 3, 1])

a = [1, 1, 1, 3] | c = [1, 1, 3, 3]
i=0, j=0: bằng 1 -> ghép, result=[1]
i=1, j=1: bằng 1 -> ghép, result=[1, 1]
i=2, j=2: 1 < 3 -> i += 1
i=3, j=2: bằng 3 -> ghép, result=[1, 1, 3]
Dừng: i=4/4, j=3/4 -> result=[1, 1, 3]


## 8. Cách 5 — Binary search trên mảng lớn (Follow up 2, biến thể)

Khi `nums1` rất nhỏ và `nums2` rất lớn **đã sắp xếp**: với mỗi giá trị `x` (đếm `k1` lần trong `nums1`), dùng `bisect_left` và `bisect_right` để biết `x` xuất hiện bao nhiêu lần trong `nums2` trong `O(log m)`, rồi lấy `min`.

`bisect_right(arr, x)` trả vị trí **ngay sau** bản sao cuối của `x` → số lần xuất hiện = `bisect_right - bisect_left`.

In [13]:
from bisect import bisect_left, bisect_right
arr = [1, 1, 3, 3, 3, 8]
for x in [1, 3, 5]:
    lo, hi = bisect_left(arr, x), bisect_right(arr, x)
    print(f"x={x}: left={lo}, right={hi}, số lần = {hi - lo}")

x=1: left=0, right=2, số lần = 2
x=3: left=2, right=5, số lần = 3
x=5: left=5, right=5, số lần = 0


In [14]:
def intersect_binary_search(nums1: list[int], nums2: list[int]) -> list[int]:
    big = sorted(nums2)
    out = []
    for x, k1 in Counter(nums1).items():       # .items() cho từng cặp (khóa, số đếm)
        k2 = bisect_right(big, x) - bisect_left(big, x)
        out.extend([x] * min(k1, k2))
    return out

print(sorted(intersect_binary_search([1, 1, 1, 3], [1, 3, 3, 1])))
print(sorted(intersect_binary_search([4, 9, 5], [9, 4, 9, 8, 4])))

[1, 1, 3]
[4, 9]


## 9. Follow up 3 — `nums2` nằm trên đĩa, bộ nhớ hạn chế

Ý tưởng: **đếm mảng nhỏ (`nums1`) trong RAM**, rồi **đọc `nums2` theo từng khúc (chunk)** và áp dụng Đoạn 2–3 của Cách 2 cho từng phần tử. Không bao giờ cần nạp toàn bộ `nums2`.

Nếu cả hai đều quá lớn: sắp xếp ngoài (external sort) từng mảng trên đĩa rồi chạy **hai con trỏ** dạng stream.

Dưới đây mô phỏng "đĩa" bằng một **generator** trả từng chunk. Generator là hàm dùng `yield` để trả dần từng giá trị thay vì trả cả list một lúc.

In [15]:
def read_chunks(data, size):
    # Mô phỏng đọc file theo từng khúc 'size' phần tử
    for start in range(0, len(data), size):
        chunk = data[start:start + size]       # slicing: lấy đoạn [start, start+size)
        print("  đọc chunk:", chunk)
        yield chunk

def intersect_stream(nums1, chunks):
    cnt = Counter(nums1)
    out = []
    for chunk in chunks:
        for y in chunk:
            if cnt[y] > 0:
                out.append(y)
                cnt[y] -= 1
    return out

print(intersect_stream([1, 1, 1, 3], read_chunks([1, 3, 3, 1], size=2)))

  đọc chunk: [1, 3]
  đọc chunk: [3, 1]
[1, 3, 1]


## 10. Code nộp LeetCode

In [16]:
class Solution:
    def intersect(self, nums1: list[int], nums2: list[int]) -> list[int]:
        if len(nums1) > len(nums2):          # đếm mảng nhỏ hơn để tiết kiệm bộ nhớ
            nums1, nums2 = nums2, nums1
        cnt = {}
        for x in nums1:
            cnt[x] = cnt.get(x, 0) + 1
        result = []
        for y in nums2:
            if cnt.get(y, 0) > 0:
                result.append(y)
                cnt[y] -= 1
        return result


sol = Solution()
print(sol.intersect([1, 2, 2, 1], [2, 2]))
print(sol.intersect([4, 9, 5], [9, 4, 9, 8, 4]))

[2, 2]
[9, 4]


`nums1, nums2 = nums2, nums1` là **hoán đổi** hai biến trong một dòng (tuple unpacking): vế phải tạo cặp `(nums2, nums1)` rồi gán lần lượt.

Bản nộp ngắn bằng `Counter`:

```python
from collections import Counter

class Solution:
    def intersect(self, nums1: list[int], nums2: list[int]) -> list[int]:
        return list((Counter(nums1) & Counter(nums2)).elements())
```

## 11. Vì sao đúng?

Gọi `f1(v)`, `f2(v)` là số lần giá trị `v` xuất hiện trong `nums1`, `nums2`.

**Invariant (Cách 2):** *sau khi xét `nums2[0..k]`, với mọi `v`: số lần `v` có trong `result` = `min(f1(v), số lần v trong nums2[0..k])`, và `cnt[v] = f1(v) − (số lần v trong result)`.*

- Ban đầu: `result` rỗng, `cnt[v] = f1(v)` → đúng.
- Gặp `y = v`: nếu `cnt[v] > 0` nghĩa là số lần đã ghép `< f1(v)` → ghép thêm vẫn `≤ f1(v)` và bằng số lần gặp → đúng `min`. Nếu `cnt[v] = 0` → đã ghép đủ `f1(v)` lần → không thêm, `min` vẫn là `f1(v)`.
- Kết thúc: mỗi `v` xuất hiện `min(f1(v), f2(v))` lần — đúng yêu cầu.

**Cách hai con trỏ:** khi `a[i] < c[j]`, mọi phần tử từ `c[j]` trở đi đều `> a[i]` nên `a[i]` không còn cặp → bỏ an toàn. Khi bằng nhau, ghép hai phần tử đó với nhau không làm mất cặp nào khác (các bản sao còn lại vẫn đứng liền nhau và tiếp tục được ghép).

## 12. Độ phức tạp

Gọi `n = len(nums1)`, `m = len(nums2)`.

| Cách | Time | Space phụ | Khi nào hợp |
|---|---|---|---|
| 1. Đánh dấu đã dùng | `O(n·m)` | `O(m)` | học khái niệm |
| 2. Hash map đếm | `O(n + m)` trung bình | `O(min(n, m))` nếu đếm mảng nhỏ | **mặc định** |
| 3. `Counter &` | `O(n + m)` | `O(n + m)` | ngắn gọn |
| 4. Sắp xếp + hai con trỏ | `O(n log n + m log m)`; `O(n + m)` nếu đã sắp | `O(n + m)` cho bản sao (`O(1)` nếu sắp tại chỗ) | Follow up 1 |
| 5. Binary search | `O(m log m + d·log m)` (`d` = số giá trị khác nhau của `nums1`) | `O(d)` | Follow up 2 khi `nums2` đã sắp |

(Không tính bộ nhớ của mảng kết quả.)

## 13. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
methods = {
    "brute": intersect_brute,
    "hashmap": intersect_hashmap,
    "counter": intersect_counter,
    "counter_explicit": intersect_counter_explicit,
    "two_pointers": intersect_two_pointers,
    "binary_search": intersect_binary_search,
    "Solution": Solution().intersect,
}
cases = [
    ("mẫu 1", [1, 2, 2, 1], [2, 2], [2, 2]),
    ("mẫu 2", [4, 9, 5], [9, 4, 9, 8, 4], [4, 9]),
    ("min lệch nhau", [1, 1, 1, 3], [1, 3, 3, 1], [1, 1, 3]),
    ("không giao", [1, 2], [3, 4], []),
    ("một phần tử", [5], [5], [5]),
    ("mảng 1 dài hơn", [2, 2, 2, 2, 2], [2, 2], [2, 2]),
    ("giá trị 0 và 1000", [0, 1000, 0], [0, 0, 0, 1000], [0, 0, 1000]),
]
for name, a, c, exp in cases:
    for mname, f in methods.items():
        res = f(list(a), list(c))
        assert sorted(res) == sorted(exp), (name, mname, res, exp)
    print(f"{name:<20} PASS -> {sorted(exp)}")
print("Mọi cách khớp nhau!")

mẫu 1                PASS -> [2, 2]
mẫu 2                PASS -> [4, 9]
min lệch nhau        PASS -> [1, 1, 3]
không giao           PASS -> []
một phần tử          PASS -> [5]
mảng 1 dài hơn       PASS -> [2, 2]
giá trị 0 và 1000    PASS -> [0, 0, 1000]
Mọi cách khớp nhau!


In [18]:
import random
rng = random.Random(350)
for _ in range(500):
    a = [rng.randint(0, 6) for _ in range(rng.randint(1, 12))]
    c = [rng.randint(0, 6) for _ in range(rng.randint(1, 12))]
    expected = sorted(intersect_brute(a, c))
    for mname, f in methods.items():
        assert sorted(f(list(a), list(c))) == expected, (mname, a, c)
print("500 test ngẫu nhiên PASS (giá trị nhỏ 0..6 để có nhiều trùng lặp)")

500 test ngẫu nhiên PASS (giá trị nhỏ 0..6 để có nhiều trùng lặp)


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Dùng set như bài 349
`list(set(nums1) & set(nums2))` mất số lần lặp: `[1,2,2,1] & [2,2]` → `[2]` thay vì `[2, 2]`.

### Lỗi 2 — Không trừ bộ đếm
Chỉ kiểm tra `y in cnt` mà không `cnt[y] -= 1` → mỗi `y` của `nums2` đều được thêm, vượt quá `f1(y)`.

### Lỗi 3 — Đọc `cnt[y]` khi `y` chưa có trong dict thường
`cnt[y]` gây `KeyError`; dùng `cnt.get(y, 0)` hoặc `Counter`.

### Lỗi 4 — Brute force quên `break`
Một `x` ghép với nhiều phần tử của `nums2`.

In [19]:
def wrong_set(nums1, nums2):
    return list(set(nums1) & set(nums2))

def wrong_no_decrement(nums1, nums2):
    cnt = {}
    for x in nums1:
        cnt[x] = cnt.get(x, 0) + 1
    return [y for y in nums2 if y in cnt]

def wrong_no_break(nums1, nums2):
    used = [False] * len(nums2)
    result = []
    for x in nums1:
        for j in range(len(nums2)):
            if not used[j] and nums2[j] == x:
                used[j] = True
                result.append(x)      # thiếu break
    return result

a, c = [1, 3], [1, 1, 3, 3, 3]
print("Đúng            :", intersect_hashmap(a, c))
print("Lỗi 1 (set)     :", wrong_set([1, 2, 2, 1], [2, 2]), "thay vì [2, 2]")
print("Lỗi 2 (không trừ):", wrong_no_decrement(a, c))
print("Lỗi 4 (thiếu break):", wrong_no_break(a, c))
try:
    {}[5]
except KeyError as e:
    print("Lỗi 3: KeyError", e)

Đúng            : [1, 3]
Lỗi 1 (set)     : [2] thay vì [2, 2]
Lỗi 2 (không trừ): [1, 1, 3, 3, 3]
Lỗi 4 (thiếu break): [1, 1, 3, 3, 3]
Lỗi 3: KeyError 5


## 15. Mẹo nhận dạng pattern & biến thể

**Nhận dạng:**
- "xuất hiện bao nhiêu lần", "giữ số lần lặp", "ghép cặp mỗi phần tử một lần" → **đếm tần suất** (`dict`/`Counter`).
- "chỉ cần biết có/không", "duy nhất" → **set** (bài 349).
- "đã sắp xếp", "không dùng thêm bộ nhớ" → **hai con trỏ**.
- "một bên nhỏ, một bên khổng lồ / trên đĩa" → đếm bên nhỏ, **stream** bên lớn.

**Biến thể liên quan:** 383 Ransom Note (đếm và trừ), 242 Valid Anagram (so hai bộ đếm), 1002 Find Common Characters (giao bội của nhiều chuỗi).

In [20]:
# Biến thể: ký tự chung (giữ số lần) của nhiều chuỗi — ý tưởng của LeetCode 1002
def common_chars(words: list[str]) -> list[str]:
    common = Counter(words[0])
    for w in words[1:]:
        common &= Counter(w)          # min từng ký tự
    return sorted(common.elements())

print(common_chars(["bella", "label", "roller"]))   # ['e', 'l', 'l']

['e', 'l', 'l']


## 16. Bài tập tự luyện (TODO)

**Bài 1 (tính tay).** Với `nums1 = [3, 3, 3, 7]`, `nums2 = [7, 3, 7, 3]`, lập bảng `cnt` sau mỗi bước của Cách 2. Kết quả là gì?

**Bài 2 (dự đoán output).** `Counter("banana") & Counter("bandana")` in ra gì?

**Bài 3 (điền chỗ trống).** Hoàn thiện Cách 2 với `Counter`.

**Bài 4 (viết lại).** Viết bản hai con trỏ giả sử `nums1`, `nums2` **đã sắp xếp** (không gọi `sorted`).

**Bài 5 (biến thể).** Trả về **số lượng** phần tử của giao bội (không cần list), dùng `O(1)` bộ nhớ phụ nếu hai mảng đã sắp.

**Bài 6 (mini challenge).** Giao bội của **k** mảng: mỗi giá trị xuất hiện `min` số lần qua tất cả mảng.

In [21]:
def exercise3_intersect_counter(nums1: list[int], nums2: list[int]) -> list[int]:
    # TODO 1: cnt = Counter(nums1)
    # TODO 2: duyệt nums2; nếu cnt[y] > 0 thì thêm y và giảm cnt[y]
    pass

# assert sorted(exercise3_intersect_counter([1, 2, 2, 1], [2, 2])) == [2, 2]

In [22]:
def exercise4_sorted_two_pointers(a: list[int], c: list[int]) -> list[int]:
    # TODO: a và c đã sắp xếp tăng dần
    pass

# assert exercise4_sorted_two_pointers([1, 1, 2, 3], [1, 2, 2, 3]) == [1, 2, 3]

In [23]:
def exercise5_count_common(a: list[int], c: list[int]) -> int:
    # TODO: chỉ đếm, không tạo list kết quả
    pass

# assert exercise5_count_common([1, 1, 2, 3], [1, 1, 1, 3]) == 3

In [24]:
def exercise6_intersect_k(arrays: list[list[int]]) -> list[int]:
    # TODO
    pass

# assert exercise6_intersect_k([[1, 1, 2, 2], [1, 2, 2, 2], [2, 2, 1, 1]]) == [1, 2, 2]

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm `nums1` trước | `cnt = {3: 3, 7: 1}` | trừ dần khi gặp trong `nums2` |
| 2 | `&` lấy min từng ký tự | đếm từng chữ của mỗi chuỗi | `b:1, a:min(3,2), n:min(2,2)` |
| 3 | giống Cách 2 | `Counter` trả 0 cho khóa thiếu | `if cnt[y] > 0: res.append(y); cnt[y] -= 1` |
| 4 | bỏ `sorted` khỏi Cách 4 | ba nhánh `<`, `>`, `==` | `==` thì tăng cả `i`, `j` |
| 5 | giống bài 4 | thay `append` bằng `count += 1` | trả về `count` |
| 6 | lặp phép `&` | bắt đầu từ `Counter(arrays[0])` | `common &= Counter(arr)` rồi `sorted(common.elements())` |

## 18. Đáp án tham khảo + Template + Checklist

**Bài 1:** `cnt = {3: 3, 7: 1}` → y=7: ghép, `{3:3, 7:0}` → y=3: ghép, `{3:2, 7:0}` → y=7: không (0) → y=3: ghép, `{3:1, 7:0}`. Kết quả `[7, 3, 3]`.

**Bài 2:** `Counter({'a': 3, 'n': 2, 'b': 1})` (banana: b1 a3 n2; bandana: b1 a3 n2 d1 → min).

In [25]:
def answer3_intersect_counter(nums1, nums2):
    cnt = Counter(nums1)
    res = []
    for y in nums2:
        if cnt[y] > 0:
            res.append(y)
            cnt[y] -= 1
    return res

def answer4_sorted_two_pointers(a, c):
    i = j = 0
    res = []
    while i < len(a) and j < len(c):
        if a[i] < c[j]:
            i += 1
        elif a[i] > c[j]:
            j += 1
        else:
            res.append(a[i]); i += 1; j += 1
    return res

def answer5_count_common(a, c):
    i = j = count = 0
    while i < len(a) and j < len(c):
        if a[i] < c[j]:
            i += 1
        elif a[i] > c[j]:
            j += 1
        else:
            count += 1; i += 1; j += 1
    return count

def answer6_intersect_k(arrays):
    common = Counter(arrays[0])
    for arr in arrays[1:]:
        common &= Counter(arr)
    return sorted(common.elements())

print("Bài 1:", trace_intersect([3, 3, 3, 7], [7, 3, 7, 3]))
print("Bài 2:", Counter("banana") & Counter("bandana"))
assert sorted(answer3_intersect_counter([1, 2, 2, 1], [2, 2])) == [2, 2]
assert answer4_sorted_two_pointers([1, 1, 2, 3], [1, 2, 2, 3]) == [1, 2, 3]
assert answer5_count_common([1, 1, 2, 3], [1, 1, 1, 3]) == 3
assert answer6_intersect_k([[1, 1, 2, 2], [1, 2, 2, 2], [2, 2, 1, 1]]) == [1, 2, 2]
print("Đáp án PASS toàn bộ!")

Sau khi đếm nums1: cnt = {3: 3, 7: 1}
Bước 1: y=7 | cnt[y] trước=1 | ghép -> giảm cnt | cnt={3: 3, 7: 0} | result=[7]
Bước 2: y=3 | cnt[y] trước=3 | ghép -> giảm cnt | cnt={3: 2, 7: 0} | result=[7, 3]
Bước 3: y=7 | cnt[y] trước=0 | không ghép (cnt = 0 hoặc không có) | cnt={3: 2, 7: 0} | result=[7, 3]
Bước 4: y=3 | cnt[y] trước=2 | ghép -> giảm cnt | cnt={3: 1, 7: 0} | result=[7, 3, 3]
Bài 1: [7, 3, 3]
Bài 2: Counter({'a': 3, 'n': 2, 'b': 1})
Đáp án PASS toàn bộ!


### Template nhớ nhanh

```python
class Solution:
    def intersect(self, nums1, nums2):
        cnt = {}
        for x in nums1:                      # 1) đếm
            cnt[x] = cnt.get(x, 0) + 1
        res = []
        for y in nums2:                      # 2) ghép + trừ
            if cnt.get(y, 0) > 0:
                res.append(y)
                cnt[y] -= 1
        return res
```

### Checklist submit
- [ ] Kết quả **giữ số lần lặp** = `min(f1, f2)`, không dùng `set`?
- [ ] Có **trừ bộ đếm** sau mỗi lần ghép?
- [ ] Dùng `cnt.get(y, 0)` hoặc `Counter` để không bị `KeyError`?
- [ ] Hai con trỏ: đã sắp xếp; khi bằng nhau tăng **cả hai** con trỏ?
- [ ] Tự test so sánh `sorted(...)` vì thứ tự tùy ý?
- [ ] Trả lời được 3 Follow up: đã sắp → hai con trỏ; lệch kích thước → đếm mảng nhỏ / binary search; trên đĩa → đếm mảng nhỏ + đọc stream.